In [ ]:
# https://www.kaggle.com/competitions/titanic/submissions
# Score: 0.76555

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from qtml import LassoNNClassifier, NN_Classifier





MODELS_CLASSIFICATION = {

    # ── KNN ──
    "KNN": (KNeighborsClassifier(),
            {"n_neighbors": range(1, 21)}, -1),

    # ── Lasso ──
    "Lasso": (LogisticRegression(solver="saga", max_iter=10000, l1_ratio=1.0),
              {"C": [0.001, 0.01, 0.1, 1]}, -1),

    # ── ElasticNet ──
    "ElasticNet": (LogisticRegression(solver="saga", max_iter=10000, l1_ratio=0.5),
                   {"C": [0.001, 0.01, 0.1, 1]}, -1),

    # ── Random Forest ──
    "RandomForest": (RandomForestClassifier(random_state=42),
                     {"n_estimators": [100, 300],
                      "max_depth": [5, 10, None],
                      "min_samples_leaf": [1, 5]}, -1),

    # ── Gradient Boosting ──
    "GBM": (GradientBoostingClassifier(random_state=42),
            {"n_estimators": [100, 300],
             "max_depth": [3, 5],
             "learning_rate": [0.01, 0.1]}, -1),

    # ── XGBoost ──
    "XGBoost": (XGBClassifier(eval_metric="logloss", random_state=42),
                {"n_estimators": [100, 300],
                 "max_depth": [3, 5, 7],
                 "learning_rate": [0.01, 0.1],
                 "subsample": [0.8, 1.0]}, -1),

    # ── LightGBM ──
    "LightGBM": (LGBMClassifier(verbose=-1, random_state=42),
                 {"n_estimators": [100, 300],
                  "max_depth": [3, 5, -1],
                  "learning_rate": [0.01, 0.1],
                  "num_leaves": [31, 63]}, -1),

    # ── LassoNN ──
    "LassoNN (L1)": (LassoNNClassifier(n_iters=100),
                     {"hidden_dims": [(64,), (64, 32)], #"hidden_dims": [(32,), (64,), (64, 32), (128, 64), (128, 64, 32)],
                      "M": [1, 10]}, 1), #  "M": [1, 5, 10]}

    # ── PyTorch NN ──
    "PyTorch NN": (NN_Classifier(),
                   {"hidden_dims": [(64,), (64, 32)], # [(32,), (64,), (64, 32), (128, 64)],
                    "lr": [0.001], # "lr": [0.001, 0.01],
                    "weight_decay": [0, 0.01],
                    "l1_lambda": [0, 0.001],
                    "dropout": [0.3, 0.4, 0.5],
                    "n_epochs": [100]}, 1),
}


'''
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr
from sklearn.preprocessing import scale
import pandas as pd
import numpy as np

# Daten aus R laden
importr("ISLR2")
ro.r('data(Caravan, package = "ISLR2")')

with localconverter(ro.default_converter + pandas2ri.converter):
    df = ro.conversion.rpy2py(ro.r("Caravan"))

# X und y vorbereiten
X = pd.DataFrame(scale(df.iloc[:, :85]), columns=df.columns[:85])
y = (df["Purchase"] == "Yes").astype(int).values
print(X.shape, y.shape)
'''

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# ── Kaggle CSVs laden ──
train = pd.read_csv("./data/train.csv")
test  = pd.read_csv("./data/test.csv")

# ── Gleiche Aufbereitung für beide ──
def prepare(df):
    df = df[["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]].copy()
    df["Age"] = df["Age"].fillna(df["Age"].median())
    df["Fare"] = df["Fare"].fillna(df["Fare"].median())
    df["Embarked"] = df["Embarked"].fillna("S")
    df = pd.get_dummies(df, columns=["Sex", "Embarked"], drop_first=True)
    return df.astype(float)

X = prepare(train)
y = train["Survived"].values
X_new = prepare(test)

scaler = StandardScaler()
X = scaler.fit_transform(X)
print(np.isnan(X).sum())  # Sollte 0 sein

X_new = scaler.transform(X_new)

print(X.shape, y.shape, X_new.shape)




# ── Trainieren + Evaluieren ──
import sys
# sys.path not needed after pip install qtml
from qtml.Cross_Sectional import run

models = MODELS_CLASSIFICATION
results = run(X, y, models)



In [ ]:
# ── Kaggle Submission ──
best_model = results["XGBoost"]  # oder welches Modell am besten war
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": best_model.predict(X_new)
})
submission.to_csv("./data/submission.csv", index=False)
print("submission.csv gespeichert!")
